We will be learning how to create a simple CNN Model in Pytorch, use it for training and testing. We will look into data loaders, optimizers too. Lets get started.

Start with all the imports

In [6]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.backends import cudnn
from torch.autograd import Variable
from PIL import Image
import csv
import os
from tqdm import tqdm
from torchvision import models



Set your working device CPU or GPU

In [3]:

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)



cpu


Create the transform object for performing transformation from PIL object to image, crop, resize, normalize

In [4]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])

Define the network

In [7]:
class Net(nn.Module):
  def __init__(self):
    super(Net, self).__init__()

    # vgg_pretrained_features = models.alexnet(pretrained=True).features
    # self.features = nn.Sequential(*list(vgg_pretrained_features.children())[:-1])
    self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=5),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

    for param in self.features.parameters():
           param.requires_grad = True
    self.classifier = nn.Linear(256, 100)

  def forward(self, x):
    x = self.features(x)
    x = x.view(x.size(0), -1)
    x = self.classifier(x)
    return x


net = Net()
net.cuda()

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

Define the optimizer and loss function

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.01,momentum=0.9, weight_decay=5e-4)

Load the data

In [ ]:

class Dldata:

    # def __init__(self, data_dir, phone='iphone'):
  def __init__(self,train=True):
    self.train=train
    if self.train:
      self.trainlist=[]
      self.trainlabels=[]

      with open('/content/drive/MyDrive/Colab Notebooks/cifar100.csv') as csv_file:
        csv_reader = csv.reader(csv_file, delimiter=',')

        for row in csv_reader:
          self.trainlist.append(row[0])
          self.trainlabels.append(row[2])



      self.image_dir = '/content/drive/MyDrive/Colab Notebooks/bumablation/bluroutmaps'




    def __len__(self):
        return len(self.trainlist)

    def __getitem__(self, idx):

        img_name = self.trainlist[idx]



        image = Image.open(os.path.join(self.image_dir, img_name))
        image = transform(image)
        label= int(self.trainlabels[idx])
        return image,label

trainset = Dldata(train=True)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64,
                                          shuffle=True, num_workers=2)

Train the network

In [ ]:
for epoch in range(3):  # loop over the dataset multiple times

  running_loss = 0.0
  for i, (images, labels) in enumerate(tqdm(trainloader, unit='batch')):

    labels=labels.long()
    images, target = Variable(images,requires_grad=True), Variable(labels)
    images,target = images.cuda(),target.cuda()
    optimizer.zero_grad()
    outputs = net(images)
    loss = criterion(outputs,target)

    loss.backward()
    optimizer.step()

Sample examples of CNN

https://www.researchgate.net/profile/Abdulaziz-Shehab-2/publication/335524166/figure/fig3/AS:797972254167041@1567262819744/Classic-Structure-of-AlexNet-Deep-Neural-Network-24.jpg


https://www.researchgate.net/publication/368805311/figure/fig5/AS:11431281122426637@1677332195139/MDB-diagram-of-the-applied-BK.png


https://www.researchgate.net/profile/Abhay-Shah-3/publication/327294790/figure/fig3/AS:676009938530304@1538184737760/Network-architecture-for-UNET-based-methods-Nnumber-of-kernels-used-in-a-given-layer.png


https://www.researchgate.net/publication/363567630/figure/fig4/AS:11431281175948850@1689963792701/Illustration-of-pre-trained-2D-Resnet50-architecture-used-for-extraction-of-deep-features.png

In [ ]:
#I you want to use MNIST dataset you will get the loader at https://www.kaggle.com/code/hojjatk/read-mnist-dataset/notebook

# a simpler use can be

!pip install idx2numpy

import idx2numpy
import numpy as np
file = '/content/t10k-images-idx3-ubyte'

arr = idx2numpy.convert_from_file(file)
print(arr.shape) # 10000 images with size 28*28 each, you can now use them like array entries.

#Remember these are already read image data, so no image reading is required here.


(10000, 28, 28)
